## Attributes per span:
  - name
  - span_kind
  - parent_id
  - start_time
  - end_time
  - status_code
  - status_message
  - events
  - context.span_id
  - context.trace_id
  - attributes.input.value
  - attributes.input.mime_type
  - attributes.output.value
  - attributes.output.mime_type
  - attributes.metadata
  - attributes.llm.invocation_parameters
  - attributes.llm.input_messages
  - attributes.llm.output_messages
  - attributes.llm.model_name
  - attributes.llm.system
  - attributes.llm.provider
  - attributes.llm.tools
  - attributes.llm.token_count.prompt
  - attributes.llm.token_count.prompt_details.cache_read
  - attributes.llm.token_count.completion_details.reasoning
  - attributes.llm.token_count.total
  - attributes.llm.token_count.completion
  - attributes.llm.token_count.completion_details.audio
  - attributes.llm.token_count.prompt_details.audio
  - attributes.openinference.span.kind
  - attributes.session.id
  - attributes.tool.description
  - attributes.tool.name

In [1]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

# Session Analyzer Example Usage

Interactive notebook version of the example script for generating cost, efficiency, and usage analytics for Phoenix sessions.


## How to Use
- Configure your environment variables (see `example.env`) before connecting to Arize.
- Update the session IDs in the code cells with values from your workspace.
- Run the setup cell once, then execute any example cell independently.


In [2]:
from session_analyzer import SessionAnalyzer

try:
    analyzer = SessionAnalyzer()
    print("✅ SessionAnalyzer ready.")
except Exception as exc:
    analyzer = None
    print(f"❌ Failed to initialize SessionAnalyzer: {exc}")


/home/wasini/Documents/pluralit/mobsta/infra/arize_reporter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ SessionAnalyzer ready.


## Example 1 – Generate All Reports

Produce the core cost, efficiency, and usage pattern reports for a single session and display summary metrics.


In [ ]:
session_id_example_1 = "postman-31bf6ed1-9b9b-4e48-a0a4-7b408f03cf25"  # Replace with your session ID

if analyzer is None:
    raise RuntimeError("Initialize the analyzer in the setup cell before running examples.")

print("=" * 80)
print("EXAMPLE 1: Generate All Reports")
print("=" * 80)
print(f"\nAnalyzing session: {session_id_example_1}")

try:
    results = analyzer.generate_all_reports(session_id_example_1)

    print("\n📊 REPORT SUMMARY")
    print("-" * 80)

    core_cost = results["core_cost"]["summary"]
    cost_est = core_cost.get("cost_estimate", {})
    print("\n💰 Core Cost Metrics:")
    print(f"  Total Cost: ${cost_est.get('total_cost_usd', 0):.4f}")
    print(f"  Total Tokens: {core_cost.get('total_tokens', 0):,}")
    print(f"  Spans: {core_cost.get('span_count', 0)}")

    efficiency = results["efficiency"]["summary"]
    cache_eff = efficiency.get("cache_efficiency", {})
    print("\n⚡ Efficiency Metrics:")
    print(f"  Cache Hit Rate: {cache_eff.get('cache_hit_rate', 0):.1%}")
    print(f"  Cache Savings: ${cache_eff.get('estimated_savings_usd', 0):.4f}")
    print(f"  Cost per Token: ${efficiency.get('cost_per_token', 0):.6f}")

    usage = results["usage_patterns"]["summary"]
    by_type = usage.get("by_span_type", {})
    print("\n📦 Usage Patterns:")
    for span_type, data in by_type.items():
        print(f"  {span_type}: {data['count']} spans, {data['tokens']:,} tokens")

    print(f"\n✅ All reports saved to outputs/{session_id_example_1}/")
except Exception as exc:
    print(f"❌ Error: {exc}")


## Example 2 – Core Cost Report Only

Generate the core cost report for a single session.


In [ ]:
session_id_example_2 = "your-session-id-here"  # Replace with your session ID

if analyzer is None:
    raise RuntimeError("Initialize the analyzer in the setup cell before running examples.")

print("=" * 80)
print("EXAMPLE 2: Generate Individual Reports")
print("=" * 80)
print(f"\nGenerating core cost report for: {session_id_example_2}")

try:
    result = analyzer.core_cost_report(session_id_example_2)
    summary = result["summary"]

    print(f"\n💰 Total Cost: ${summary['cost_estimate']['total_cost_usd']:.4f}")
    print(f"📊 Total Tokens: {summary['total_tokens']:,}")

    if summary.get("by_model"):
        print("\nCost by Model:")
        for model, data in summary["by_model"].items():
            print(f"  • {model}: ${data['cost_usd']:.4f} ({data['tokens']:,} tokens)")

    print(f"
Report saved to: {result['files']['summary_file']}")
except Exception as exc:
    print(f"❌ Error: {exc}")


## Example 3 – Fetch Custom Attributes

Fetch spans with a custom list of attributes for ad-hoc analysis.


In [ ]:
session_id_example_3 = "your-session-id-here"  # Replace with your session ID
attributes_example_3 = [
    "name",
    "llm.model_name",
    "llm.token_count.total",
    "llm.token_count.prompt",
    "llm.token_count.completion",
    "start_time",
    "end_time",
]

if analyzer is None:
    raise RuntimeError("Initialize the analyzer in the setup cell before running examples.")

print("=" * 80)
print("EXAMPLE 3: Custom Span Fetching")
print("=" * 80)
print(f"\nFetching custom attributes for: {session_id_example_3}")

try:
    df = analyzer.fetch_session_spans(
        session_id_example_3,
        attributes=attributes_example_3,
    )

    print(f"\n✅ Fetched {len(df)} spans with {len(df.columns)} columns")

    if not df.empty and "llm.token_count.total" in df.columns:
        total_tokens = df["llm.token_count.total"].sum()
        avg_tokens = df["llm.token_count.total"].mean()

        print("\nCustom Analysis:")
        print(f"  Total tokens: {total_tokens:,.0f}")
        print(f"  Average tokens per span: {avg_tokens:.0f}")

        if "name" in df.columns:
            max_idx = df["llm.token_count.total"].idxmax()
            max_span = df.loc[max_idx]
            print(f"  Most expensive span: {max_span['name']} ({max_span['llm.token_count.total']:,.0f} tokens)")
except Exception as exc:
    print(f"❌ Error: {exc}")


## Example 4 – Compare Multiple Sessions

Compare key metrics across a list of sessions.


In [ ]:
session_ids_example_4 = [
    "session-1",
    "session-2",
    "session-3",
]  # Replace with your session IDs

if analyzer is None:
    raise RuntimeError("Initialize the analyzer in the setup cell before running examples.")

print("=" * 80)
print("EXAMPLE 4: Compare Multiple Sessions")
print("=" * 80)
print(f"\nComparing {len(session_ids_example_4)} sessions...\n")

try:
    comparison_data = []

    for session_id in session_ids_example_4:
        result = analyzer.core_cost_report(session_id)
        summary = result["summary"]

        comparison_data.append({
            "session_id": session_id[:30],
            "cost": summary["cost_estimate"]["total_cost_usd"],
            "tokens": summary["total_tokens"],
            "spans": summary["span_count"],
        })

    print(f"{'Session ID':32} {'Cost':>12} {'Tokens':>12} {'Spans':>8}")
    print("-" * 70)
    for data in comparison_data:
        print(f"{data['session_id']:32} ${data['cost']:11.4f} {data['tokens']:12,} {data['spans']:8}")

    total_cost = sum(d["cost"] for d in comparison_data)
    total_tokens = sum(d["tokens"] for d in comparison_data)
    total_spans = sum(d["spans"] for d in comparison_data)

    print("-" * 70)
    print(f"{'TOTAL':32} ${total_cost:11.4f} {total_tokens:12,} {total_spans:8}")
except Exception as exc:
    print(f"❌ Error: {exc}")


## Example 5 – Fetch All Attributes

Fetch every available column for a session to support exploratory analysis.


In [6]:
session_id_example_5 = "stream-14261982-1f8d-4821-9009-c35a65e55353"  # Replace with your session ID

if analyzer is None:
    raise RuntimeError("Initialize the analyzer in the setup cell before running examples.")

print("=" * 80)
print("EXAMPLE 5: Fetch All Attributes")
print("=" * 80)
print(f"\nFetching all available attributes for: {session_id_example_5}")

try:
    # df = analyzer.fetch_session_spans(session_id=session_id_example_5)
    
    res = analyzer.get_messages(session_id=session_id_example_5, project_name="mobsta-production")

    print(f"\n✅ Fetched {res["row_count"]} spans")
except Exception as exc:
    print(f"❌ Error: {exc}")


EXAMPLE 5: Fetch All Attributes

Fetching all available attributes for: stream-14261982-1f8d-4821-9009-c35a65e55353

✅ Fetched 47 spans


## Example 6 – Conversation Extraction (Single Session)

Extract agent or tool conversations for a specific session and inspect the resulting CSV.


In [6]:
import pandas as pd

session_id_example_6 = "postman-31bf6ed1-9b9b-4e48-a0a4-7b408f03cf25"  # Replace with your session ID

if analyzer is None:
    raise RuntimeError("Initialize the analyzer in the setup cell before running examples.")

print("=" * 80)
print("EXAMPLE 6: Conversation Extraction")
print("=" * 80)
print(f"\nExtracting conversations for: {session_id_example_6}")

try:
    result = analyzer.extract_conversation_data(session_id=session_id_example_6)

    print("\n✅ Extraction complete!")
    print(f"  Total conversations: {result['summary']['total_conversations']}")
    print(f"  Span type: {result['span_kind']}")
    print(f"  CSV file: {result['files']['csv_file']}")

    df = pd.read_csv(result["files"]["csv_file"])
    print("\n📊 Extracted Data:")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {', '.join(df.columns.tolist())}")

    if len(df) > 0:
        sample = df.iloc[0]
        print("\nSample row (first conversation):")
        print(f"  Session: {sample['session_id']}")
        print(f"  Start time: {sample['start_time']}")
        print(f"  Span ID: {sample['span_id']}")
except Exception as exc:
    print(f"❌ Error: {exc}")


EXAMPLE 6: Conversation Extraction

Extracting conversations for: postman-31bf6ed1-9b9b-4e48-a0a4-7b408f03cf25
first human content: hi
last human content: how can I help?
first AI LangChain message: content='hello' additional_kwargs={} response_metadata={}
last human LangChain message: content='how can I help?' additional_kwargs={} response_metadata={}

✅ Extraction complete!
  Total conversations: 37
  Span type: AGENT
  CSV file: outputs/postman-31bf6ed1-9b9b-4e48-a0a4-7b408f03cf25/conversations_postman-31bf6ed1-9b9b-4e48-a0a4-7b408f03cf25_20251106_143130.csv

📊 Extracted Data:
  Rows: 37
  Columns: human, ai, start_time, end_time, session_id, span_id, metadata

Sample row (first conversation):
  Session: postman-31bf6ed1-9b9b-4e48-a0a4-7b408f03cf25
  Start time: 2025-11-03 17:02:28.162865+00:00
  Span ID: 53717a99943cfd62


## Example 6B – Conversation Extraction by Time Range

Extract conversations across all sessions for the last 24 hours (configurable).


In [ ]:
from datetime import datetime, timedelta

if analyzer is None:
    raise RuntimeError("Initialize the analyzer in the setup cell before running examples.")

print("=" * 80)
print("EXAMPLE 6B: Conversation Extraction - Time Range")
print("=" * 80)
print("\nExtracting conversations from the last 24 hours...")

try:
    result = analyzer.extract_conversation_data()

    print("\n✅ Extraction complete!")
    print(f"  Total conversations: {result['summary']['total_conversations']}")
    print(f"  Sessions analyzed: {result['summary']['sessions_analyzed']}")
    print(f"  CSV file: {result['files']['csv_file']}")

except Exception as exc:
    print(f"❌ Error: {exc}")

# Customize the time window by passing start_time/end_time parameters, for example:
# start_time = datetime.utcnow() - timedelta(days=3)
# result = analyzer.extract_conversation_data(start_time=start_time)


## Example 6C – Conversation Extraction by Span Type

Extract a different span kind (e.g., TOOL instead of AGENT) for a session.


In [ ]:
session_id_example_6c = "postman-31bf6ed1-9b9b-4e48-a0a4-7b408f03cf25"  # Replace with your session ID
span_kind_example_6c = "TOOL"  # Try other values such as "AGENT"

if analyzer is None:
    raise RuntimeError("Initialize the analyzer in the setup cell before running examples.")

print("=" * 80)
print("EXAMPLE 6C: Conversation Extraction - Span Type")
print("=" * 80)
print(f"\nExtracting {span_kind_example_6c} conversations for: {session_id_example_6c}")

try:
    result = analyzer.extract_conversation_data(
        session_id=session_id_example_6c,
        span_kind=span_kind_example_6c,
    )

    print("\n✅ Extraction complete!")
    print(f"  Total {span_kind_example_6c} conversations: {result['summary']['total_conversations']}")
    print(f"  Span type: {result['span_kind']}")
    print(f"  CSV file: {result['files']['csv_file']}")
except Exception as exc:
    print(f"❌ Error: {exc}")
